In [0]:
%sql  
CREATE VOLUME IF NOT EXISTS ecommerce_analytics.default.data_feed   
COMMENT 'Managed volume for raw file ingestion';

In [0]:
path = "/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv"

df = spark.read.csv(path, header=True, inferSchema=True)
display(df)

order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
10001,2022-01-01,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.2
10002,2022-01-02,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.1
10003,2022-01-03,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35
10004,2022-01-04,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.9
10005,2022-01-05,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56
10006,2022-01-06,1071,Electronics,West,2,264.44,0.2,Card,3,3.3,423.1
10007,2022-01-07,1700,Clothing,South,6,564.6,0.23,Card,1,1.8,2608.45
10008,2022-01-08,1020,Beauty,North,6,265.71,0.13,Card,2,3.7,1387.01
10009,2022-01-09,1614,Clothing,South,6,224.65,0.27,Card,2,1.4,983.97
10010,2022-01-10,1121,Clothing,West,4,272.71,0.06,Card,10,4.5,1025.39


In [0]:
from pyspark.sql.functions import current_timestamp, column 

#Define Source and Target Path 
source_path = "/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv"
target_path = "ecommerce_analytics.bronze.ecommerce_sales"

df = (
    spark.read.format('csv')
   .option("header","true")
   .option("inferSchema","true")
   .load(source_path)
   .withColumn("_ingested_at",current_timestamp())
   .withColumn("_source_file",column("_metadata.file_path"))
)

df.write.format("delta").mode("append").saveAsTable("ecommerce_analytics.bronze.ecommerce_sales")

In [0]:
#Second Method of Doing this
%sql
CREATE TABLE IF NOT EXISTS ecommerce_analytics.bronze.ecommerce_sales;

In [0]:
%sql
COPY INTO ecommerce_analytics.bronze.ecommerce_sales
FROM (
    SELECT 
        *,
        current_timestamp() AS _ingested_at,
        _metadata.file_path AS _source_file
    FROM '/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
5000,5000,0


In [0]:
%sql
select * from ecommerce_analytics.bronze.ecommerce_sales;

order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue,_ingested_at,_source_file
10001,2022-01-01,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.2,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv
10002,2022-01-02,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.1,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv
10003,2022-01-03,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv
10004,2022-01-04,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.9,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv
10005,2022-01-05,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv
10006,2022-01-06,1071,Electronics,West,2,264.44,0.2,Card,3,3.3,423.1,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv
10007,2022-01-07,1700,Clothing,South,6,564.6,0.23,Card,1,1.8,2608.45,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv
10008,2022-01-08,1020,Beauty,North,6,265.71,0.13,Card,2,3.7,1387.01,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv
10009,2022-01-09,1614,Clothing,South,6,224.65,0.27,Card,2,1.4,983.97,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv
10010,2022-01-10,1121,Clothing,West,4,272.71,0.06,Card,10,4.5,1025.39,2026-11-13T19:11:56.002Z,dbfs:/Volumes/ecommerce_analytics/default/data_feed/ecommerce_sales_analytics_5000.csv


In [0]:
%sql
--Region wise 
select region, quantity, unit_price, discount,revenue 
from ecommerce_analytics.bronze.ecommerce_sales;

--Product Category and Payment Method Wise  
select product_category, payment_method, quantity, unit_price, revenue, delivery_days,customer_rating
from ecommerce_analytics.bronze.ecommerce_sales;

product_category,payment_method,quantity,unit_price,revenue,delivery_days,customer_rating
Beauty,Wallet,7,373.65,1883.2,10,4.7
Clothing,Card,7,47.74,304.1,6,3.9
Beauty,COD,3,311.28,644.35,6,2.5
Electronics,Wallet,5,524.47,2569.9,6,1.6
Clothing,Wallet,5,139.87,468.56,4,4.9
Electronics,Card,2,264.44,423.1,3,3.3
Clothing,Card,6,564.6,2608.45,1,1.8
Beauty,Card,6,265.71,1387.01,2,3.7
Clothing,Card,6,224.65,983.97,2,1.4
Clothing,Card,4,272.71,1025.39,10,4.5


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_analytics.silver.region_analytics 
AS 
select region, quantity, unit_price, discount,revenue, current_timestamp() AS _ingested_at
from ecommerce_analytics.bronze.ecommerce_sales;

region,quantity,unit_price,discount,revenue,_ingested_at
South,7,373.65,0.28,1883.2,2026-11-14T09:41:33.629Z
South,7,47.74,0.09,304.1,2026-11-14T09:41:33.629Z
East,3,311.28,0.31,644.35,2026-11-14T09:41:33.629Z
West,5,524.47,0.02,2569.9,2026-11-14T09:41:33.629Z
West,5,139.87,0.33,468.56,2026-11-14T09:41:33.629Z
West,2,264.44,0.2,423.1,2026-11-14T09:41:33.629Z
South,6,564.6,0.23,2608.45,2026-11-14T09:41:33.629Z
North,6,265.71,0.13,1387.01,2026-11-14T09:41:33.629Z
South,6,224.65,0.27,983.97,2026-11-14T09:41:33.629Z
West,4,272.71,0.06,1025.39,2026-11-14T09:41:33.629Z


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_analytics.silver.product_payment_analytics 
AS 
select product_category, payment_method, quantity, unit_price, revenue, delivery_days,customer_rating, current_timestamp() AS _ingested_at
from ecommerce_analytics.bronze.ecommerce_sales;

num_affected_rows,num_inserted_rows


In [0]:
%sql 
select product_category, sum(quantity) as Total_quantity,round(sum(revenue),2) as Total_revenue
from ecommerce_analytics.silver.product_payment_analytics
group by product_category;


product_category,Total_quantity,Total_revenue
Beauty,2995,765860.88
Clothing,6171,1531931.72
Electronics,7109,1829899.22
Home,3949,982083.92


In [0]:
%sql
select payment_method, round(sum(revenue),2) as Total_revenue
from ecommerce_analytics.silver.product_payment_analytics
group by payment_method;

payment_method,Total_revenue
Wallet,955119.1
Card,2366248.2
COD,1788408.44


In [0]:
%sql
select distinct product_category, delivery_days
from ecommerce_analytics.silver.product_payment_analytics
order by delivery_days desc 
limit 5;

product_category,delivery_days
Beauty,11
Clothing,11
Electronics,11
Home,11
Clothing,10


In [0]:
%sql
select region, sum(quantity) as Quantity
from ecommerce_analytics.silver.region_analytics
group by region
limit 3;

region,Quantity
South,4884
East,4916
West,5260


In [0]:
%sql
select region, round(sum(revenue),2) as Total_Revenue
from ecommerce_analytics.silver.region_analytics
group by region
limit 3;

region,Total_Revenue
South,1246640.9
East,1236044.23
West,1345582.16


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_analytics.gold.product_analysis 
AS 
select product_category, sum(quantity) as Total_quantity,round(sum(revenue),2) as Total_revenue
from ecommerce_analytics.silver.product_payment_analytics
group by product_category;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_analytics.gold.payment_analysis 
AS 
select payment_method, round(sum(revenue),2) as Total_revenue
from ecommerce_analytics.silver.product_payment_analytics
group by payment_method;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_analytics.gold.best_performing_region 
AS 
select region, round(sum(revenue),2) as Total_Revenue
from ecommerce_analytics.silver.region_analytics
group by region
limit 3;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_analytics.gold.region_quantity_analysis 
AS 
select region, sum(quantity) as Quantity
from ecommerce_analytics.silver.region_analytics
group by region
limit 3;

num_affected_rows,num_inserted_rows


In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce_analytics.gold.products_with_best_delivery 
AS 
select distinct product_category, delivery_days
from ecommerce_analytics.silver.product_payment_analytics
order by delivery_days desc 
limit 5;

num_affected_rows,num_inserted_rows
